In [3]:
import os
import re
import math
import shutil
import subprocess

from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from matplotlib.dates import DateFormatter
from dateutil.tz import tzutc, tzlocal
from scipy import stats
from scipy.optimize import brentq, curve_fit, fsolve

In [5]:
%load_ext autoreload
%autoreload 2

import official_GSSHA_Python_functions as gf

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:

# Get the current working directory
cwd = os.getcwd()
script_dir = Path.cwd()  # Use  if you're in Jupyter or interactive session

GSSHA_prj_name = "Waiahole_FIM_Model"
model_dir = script_dir / GSSHA_prj_name
GSSHA_executables = script_dir / 'GSSHA_applications'


### LOW FLOW ###
FLOW_TEST = [1, 50, 200, 400]
low_flow_time = 500
high_flow_time = 360


for flow in FLOW_TEST:
    new_value = float(flow)
    
    gf.replace_value_in_gssha_file(read_dir=model_dir , gssha_sample_file = "Waiahole_FIM_Model_sample_file_bc.tsf", save_filename = "Waiahole_FIM_Model_bc.tsf", new_value = new_value)

    gf.replace_value_in_gssha_file(read_dir= model_dir, gssha_sample_file = "Waiahole_FIM_Model_sample_file.xys", save_filename = "Waiahole_FIM_Model.xys", new_value = new_value)

    df_prj = read_prj_file(GSSHA_prj_name +".prj", prj_folder_path = model_dir)
  
    ows = df_prj.Value['OVERLAND_WSE'].split('\\')[:-1]
    ows.append(GSSHA_prj_name + "_flowFIMTEST_USGSgauge_" + str(flow) + ".ows")
    ows = '\\'.join(ows) + '"'

        #.otl(flow output)
    otl = df_prj.Value['OUTLET_HYDRO'].split('\\')[:-1]
    otl.append(GSSHA_prj_name + "_flowGSSHATimingTEST_streamoutlet" + str(flow) + ".otl")
    otl = '\\'.join(otl) + '"'
    df_prj.Value['OVERLAND_WSE'] = ows
    df_prj.Value['OUTLET_HYDRO'] = otl

    if flow <= 50:
        tot_time = low_flow_time
        df_prj.Value['TOT_TIME'] = tot_time
    else:
        tot_time = high_flow_time
        df_prj.Value['TOT_TIME'] = tot_time
    
    df_prj = df_prj.reset_index()
    
    gf.convert_df_to_prj(
        df_prj,
        output_folder= model_dir,
        prj_file_name=GSSHA_prj_name
    )
    
    # copy_gssha_apps_to_model(GSSHA_executables, model_dir)
    # run_gssha(model_dir, PROJECT_FILE)
    # # move_and_rename_gssha_output(MODEL_DIR, RESULTS_DIR, output_description = "TEST_RUN" + first_date, extension = "otl")
    # cleanup_model_dir(model_dir)




FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\bgorberg\\Documents\\GitHub\\Flood_Stage_Maps\\RUN_GSSHA\\Waiahole_FIM_Model\\Waiahole_FIM_Model_sample_file_bc.tsf'